# 02 - Shipping & Order Context Features

EDA found `Shipping Mode` is the single strongest predictor found so far (57pp spread in late rate), and flagged a multicollinearity risk with `Days for shipment (scheduled)` that needed checking before finalizing the feature set. This notebook builds the planned shipping/order features and resolves that check directly.




## Setup

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_IN = Path('../../../data/processed/features_step1_train.csv')
VAL_IN = Path('../../../data/processed/features_step1_val.csv')
TEST_IN = Path('../../../data/processed/features_step1_test.csv')

TRAIN_OUT = Path('../../../data/processed/features_step2_train.csv')
VAL_OUT = Path('../../../data/processed/features_step2_val.csv')
TEST_OUT = Path('../../../data/processed/features_step2_test.csv')

train_df = pd.read_csv(TRAIN_IN)
val_df = pd.read_csv(VAL_IN)
test_df = pd.read_csv(TEST_IN)


## 1. Resolve the Shipping Mode / Days for shipment (scheduled) multicollinearity flag

Check whether each `Shipping Mode` always maps to the same scheduled-days value (near-deterministic relationship) using the training data only.


In [9]:
crosstab = pd.crosstab(train_df['Shipping Mode'], train_df['Days for shipment (scheduled)'])
crosstab


Days for shipment (scheduled),0,1,2,4
Shipping Mode,,,,
First Class,0,7049,0,0
Same Day,2478,0,0,0
Second Class,0,0,8929,0
Standard Class,0,0,0,27570


**What we found:** the relationship is a **perfect 1:1 mapping**, not just near deterministic. Every order follows First Class↔1 day, Same Day↔0 days, Second Class↔2 days, Standard Class↔4 days, with zero exceptions. `Days for shipment (scheduled)` carries no information beyond what `Shipping Mode` already encodes, and keeping both would destabilize Logistic Regression's coefficients, undermining the interpretability we're relying on it for.

**Decision: drop `Days for shipment (scheduled)`, but only after using it to build `sales_per_scheduled_day` below** - this ordering matters, since the engineered feature needs the raw column to exist first.


## 2. Build the planned order/shipping context features (using Days for shipment before it's dropped)

In [10]:
def add_shipping_features(df):
    df = df.copy()
    df['sales_per_scheduled_day'] = df['Sales'] / df['Days for shipment (scheduled)'].replace(0, np.nan)
    df['sales_per_scheduled_day'] = df['sales_per_scheduled_day'].fillna(df['Sales'])  # 0-day orders: use raw sales

    df['is_express_shipping'] = df['Shipping Mode'].isin(['First Class', 'Same Day']).astype(int)

    return df

train_df = add_shipping_features(train_df)
val_df = add_shipping_features(val_df)
test_df = add_shipping_features(test_df)

print("Express shipping late rate (train):")
print(train_df.groupby('is_express_shipping')['Late_delivery_risk'].mean())


Express shipping late rate (train):
is_express_shipping
0    0.477027
1    0.822085
Name: Late_delivery_risk, dtype: float64


**What we found:** a large, meaningful gap: 47.70% late rate for non-express orders vs. 82.21% for express orders (First Class + Same Day combined), nearly a 35-percentage-point difference. Consistent with EDA's finding that First Class and Same Day both sit well above the other two modes individually.


## 3. Now drop Days for shipment (scheduled)

Safe to drop now - `sales_per_scheduled_day` has already been computed from it, and `Shipping Mode`'s one hot encoding (Notebook 04) fully captures the categorical information.


In [11]:
train_df = train_df.drop(columns=['Days for shipment (scheduled)'])
val_df = val_df.drop(columns=['Days for shipment (scheduled)'])
test_df = test_df.drop(columns=['Days for shipment (scheduled)'])

print("Dropped Days for shipment (scheduled).")
print(f"Confirm engineered features present: {[c for c in train_df.columns if c in ['sales_per_scheduled_day', 'is_express_shipping']]}")


Dropped Days for shipment (scheduled).
Confirm engineered features present: ['sales_per_scheduled_day', 'is_express_shipping']


## 4. Save

In [12]:
train_df.to_csv(TRAIN_OUT, index=False)
val_df.to_csv(VAL_OUT, index=False)
test_df.to_csv(TEST_OUT, index=False)
print("Saved step 2 outputs.")


Saved step 2 outputs.


**`DECISION_LOG.md`:** confirmed perfect (not just near) collinearity between Shipping Mode and Days for shipment (scheduled); dropped the latter after using it to build `sales_per_scheduled_day`. Confirmed `sales_per_scheduled_day` and `is_express_shipping` are present in the saved output before moving to Notebook 03.
